In [1]:
# Bias correct OSDMA8

In [2]:
import os
import xarray as xr
import warnings
from utils.utils import adjust_longitude
from utils.utils import bilinear_interp
from utils.utils import get_scenario_config
from utils.utils import standardise_latlon

In [3]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

OSDMA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8/"
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")

    # Final year is not be complete due to SH Jan-Mar missing
    dates = f"{years.start}-{years.stop - 1}"

    osdma8_file = f"OSDMA8_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    osdma8_path = os.path.join(OSDMA8_DIR, osdma8_file)
    osdma8 = xr.open_dataarray(osdma8_path)

    hist_file = f"OSDMA8_{model}_hist_01_1990-2008.nc"
    hist_path = os.path.join(OSDMA8_DIR, hist_file)
    hist = xr.open_dataarray(hist_path)

    obs_file = "Delang_BME_OSDMA8_1990_2017.nc"
    obs_path = os.path.join(OBS_DIR, obs_file)
    obs = xr.open_dataset(obs_path)["ozone"]

    # Baseline years for fi_2000 and historical
    base = slice("1990", "2008")
    hist_base = hist.sel(year=base).mean("year")
    obs_base = obs.sel(year=base).mean("year")
    # Change lat/lon coordinate names to lat, lon
    obs_base = standardise_latlon(obs_base)

    # Calculate delta
    delta_fi = adjust_longitude(osdma8 / hist_base)

    # Interpolate to the new grid
    regridder = bilinear_interp(delta_fi, obs_base)
    ds_delta_fi = regridder(delta_fi)

    # Bias correct delta
    bc_osdma8 = obs_base * ds_delta_fi

    out_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving to {out_path}")
    description = ("OSDMA8: Highest seasonal (6-month) average of 8-hour "
                   "daily maximum ozone concentrations across 15 months "
                   "(Jan-Mar) bias corrected to DeLang et al. (2021) "
                   "observations - scripts by A.F. Wells (2025)")
    bc_osdma8.attrs["description"] = description
    bc_osdma8.attrs["ensemble_number"] = ens_num
    bc_osdma8.attrs["scenario"] = scenario
    bc_osdma8.attrs["model"] = model
    bc_osdma8.attrs["units"] = "ppb"
    bc_osdma8.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245_G6, Ensemble 01
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/OSDMA8_BC_CESM2_SSP245_G6_01_2020-2083.nc
Processing SSP245_G6, Ensemble 02
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/OSDMA8_BC_CESM2_SSP245_G6_02_2020-2083.nc
Processing SSP245_G6, Ensemble 03
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/OSDMA8_BC_CESM2_SSP245_G6_03_2020-2083.nc
All processing complete.
